# Description

We evaluate the scalability of OPERON on the previously generated data with growing number of input variables.

In [1]:
# operon_scalability_poly2.py

from __future__ import annotations

import csv
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import h5py
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from pyoperon.sklearn import SymbolicRegressor

from config.scalability_config import DataCFG, OperonCFG


def _select_best_by_key(reg: SymbolicRegressor, key: str) -> Dict[str, Any]:
    front: List[Dict[str, Any]] = reg.pareto_front_
    return min(front, key=lambda d: d[key])


def _get_expression_string(reg: SymbolicRegressor, best_stats: Dict[str, Any]) -> str:
    model = best_stats.get("model", None)
    if model is None:
        model = best_stats.get("tree", None)

    if isinstance(model, str):
        return model.strip()

    return reg.get_model_string(model, precision=OperonCFG.sym_decimals)


def _operon_string_to_sympy(expr_str: str, d: int) -> Tuple[sp.Expr, List[sp.Symbol]]:
    s = expr_str.replace("^", "**")

    xs = [sp.Symbol(f"X{i+1}", real=True) for i in range(d)]
    local_dict: Dict[str, Any] = {f"X{i+1}": xs[i] for i in range(d)}

    def square(u):
        return u**2

    local_dict["square"] = square

    expr = sp.sympify(s, locals=local_dict)
    return expr, xs


def _eval_sympy(expr: sp.Expr, xs: List[sp.Symbol], X: np.ndarray) -> np.ndarray:
    f = sp.lambdify(xs, expr, modules="numpy")
    cols = [np.asarray(X[:, i], dtype=np.float64) for i in range(len(xs))]
    y = f(*cols)
    return np.asarray(y, dtype=np.float64).reshape(-1)


def _print_run_header(Expression: str, d: int, seed: int, run_j: int, n_runs: int, Xtr: np.ndarray, Xte: np.ndarray) -> None:
    print("\n" + "=" * 110)
    print(f"RUN START | Expression={Expression} | d={d} | run={run_j+1}/{n_runs} | seed={seed}")
    print(f"  X_train: {Xtr.shape} | X_test: {Xte.shape}")
    print(
        "  operon_cfg: "
        f"gens={OperonCFG.generations}, pop={OperonCFG.population_size}, "
        f"max_len={OperonCFG.max_length}, max_depth={OperonCFG.max_depth}, "
        f"allowed_symbols='{OperonCFG.allowed_symbols}', objectives={list(OperonCFG.objectives)}, "
        f"select='{OperonCFG.model_selection_criterion}'"
    )
    print("=" * 110)


def _make_seeds() -> Tuple[int, ...]:
    if len(OperonCFG.seeds) >= OperonCFG.runs_per_d:
        return OperonCFG.seeds[: OperonCFG.runs_per_d]
    return tuple(range(OperonCFG.runs_per_d))


def _aggregate(rows: List[Dict[str, Any]], Expression: str, d_list: List[int]) -> Dict[int, Dict[str, float]]:
    out: Dict[int, Dict[str, float]] = {}
    for d in d_list:
        sub = [r for r in rows if r["Expression"] == Expression and r["d"] == d]
        tr = np.array([r["train_mse"] for r in sub], dtype=np.float64)
        te = np.array([r["test_mse"] for r in sub], dtype=np.float64)
        out[d] = {
            "train_mean": float(tr.mean()),
            "train_std": float(tr.std(ddof=0)),
            "test_mean": float(te.mean()),
            "test_std": float(te.std(ddof=0)),
        }
    return out


def _plot_errorbars_uniform_x(Expression: str, agg: Dict[int, Dict[str, float]], d_list: List[int]) -> None:
    Y_LOG_SCALE = True
    X_LABEL = "Number of Input Variables"
    Y_LABEL = "MSE"

    x_pos = np.arange(len(d_list), dtype=np.int32)

    tr_mean = np.array([agg[d]["train_mean"] for d in d_list], dtype=np.float64)
    tr_std = np.array([agg[d]["train_std"] for d in d_list], dtype=np.float64)
    te_mean = np.array([agg[d]["test_mean"] for d in d_list], dtype=np.float64)
    te_std = np.array([agg[d]["test_std"] for d in d_list], dtype=np.float64)

    plt.figure()
    plt.errorbar(x_pos, tr_mean, yerr=tr_std, fmt="-o", capsize=4, label="Train MSE (mean ± std)")
    plt.errorbar(x_pos, te_mean, yerr=te_std, fmt="-o", capsize=4, label="Test MSE (mean ± std)")

    if Y_LOG_SCALE:
        plt.yscale("log")

    plt.xlabel(X_LABEL)
    plt.ylabel(Y_LABEL)
    plt.title(f"Operon scalability | Expression={Expression} | {len(d_list)} d-values")

    plt.xticks(x_pos, [str(d) for d in d_list])
    plt.legend()
    plt.tight_layout()
    plt.show()


def _write_report_csv(rows: List[Dict[str, Any]], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)

    fieldnames = [
        "Expression",
        "d",
        "run",
        "seed",
        "train_mse",
        "test_mse",
        "run_time_s",
        "expr",
    ]

    with path.open("w", newline="", encoding="utf-8") as fp:
        w = csv.DictWriter(fp, fieldnames=fieldnames)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in fieldnames})

    print(f"Saved per-run report: {str(path)}")


def main():
    t0_all = time.perf_counter()
    rows: List[Dict[str, Any]] = []

    h5_path = str(DataCFG.out_path)

    Expressions: Tuple[str, ...] = OperonCFG.kinds
    if len(Expressions) == 0:
        Expressions = ("P2",)
    else:
        Expressions = tuple("P2" if e == "P3" else e for e in Expressions)

    seeds = _make_seeds()
    n_runs = len(seeds)

    selection_key = OperonCFG.model_selection_criterion

    p = Path(OperonCFG.report_csv_path)
    report_csv_path = p.with_name(p.stem + "_poly2" + p.suffix)

    print(f"Opening HDF5: {h5_path}")
    with h5py.File(h5_path, "r") as f:
        d_list = list(map(int, f.attrs["d_list"]))

        print(f"d_list = {d_list}")
        print(f"Expressions = {list(Expressions)}")
        print(f"seeds (runs) = {list(seeds)} (runs_per_d={n_runs})")
        print(f"report_csv_path = {report_csv_path}")

        for Expression in Expressions:
            print(f"\n===== Expression: {Expression} =====")
            for d in d_list:
                print(f"Loading data for Expression={Expression}, d={d} ...")

                Xtr = f[f"data/{Expression}/d{d}/train/X"][...].astype(np.float64, copy=False)
                ytr = f[f"data/{Expression}/d{d}/train/y"][...].astype(np.float64, copy=False).reshape(-1)

                Xte = f[f"data/{Expression}/d{d}/test/X"][...].astype(np.float64, copy=False)
                yte = f[f"data/{Expression}/d{d}/test/y"][...].astype(np.float64, copy=False).reshape(-1)

                print(f"Loaded: Xtr={Xtr.shape}, ytr={ytr.shape}, Xte={Xte.shape}, yte={yte.shape}")

                for run_j, seed in enumerate(seeds):
                    _print_run_header(Expression, d, seed, run_j, n_runs, Xtr, Xte)

                    reg = SymbolicRegressor(
                        allowed_symbols=OperonCFG.allowed_symbols,
                        generations=OperonCFG.generations,
                        population_size=OperonCFG.population_size,
                        max_length=OperonCFG.max_length,
                        max_depth=OperonCFG.max_depth,
                        objectives=list(OperonCFG.objectives),
                        model_selection_criterion=OperonCFG.model_selection_criterion,
                        n_threads=OperonCFG.n_threads,
                        random_state=seed,
                    )

                    t0_run = time.perf_counter()

                    print("fit() started ...")
                    reg.fit(Xtr, ytr)
                    print("fit() finished")

                    best_stats = _select_best_by_key(reg, key=selection_key)

                    expr_str = _get_expression_string(reg, best_stats)
                    if OperonCFG.print_best_expression:
                        print("expr_str:", expr_str)

                    expr_sym, xs_sym = _operon_string_to_sympy(expr_str, d=d)
                    expr_sym = sp.simplify(expr_sym)

                    yhat_tr = _eval_sympy(expr_sym, xs_sym, Xtr)
                    yhat_te = _eval_sympy(expr_sym, xs_sym, Xte)

                    tr_mse = float(np.mean((ytr - yhat_tr) ** 2))
                    te_mse = float(np.mean((yte - yhat_te) ** 2))

                    run_s = time.perf_counter() - t0_run

                    rows.append(
                        {
                            "Expression": Expression,
                            "d": int(d),
                            "run": int(run_j),
                            "seed": int(seed),
                            "train_mse": float(tr_mse),
                            "test_mse": float(te_mse),
                            "run_time_s": float(run_s),
                            "expr": expr_str,
                        }
                    )

                    print(
                        f"RESULT | Expression={Expression} | d={d:>3} | run={run_j+1}/{n_runs} | seed={seed} | "
                        f"train_mse={tr_mse:.6e} | test_mse={te_mse:.6e} | run_time_s={run_s:.2f}"
                    )

    _write_report_csv(rows, report_csv_path)

    print("\n=== Summary (mean±std over runs) ===")
    for Expression in Expressions:
        d_list_sorted = sorted({r["d"] for r in rows if r["Expression"] == Expression})
        agg = _aggregate(rows, Expression=Expression, d_list=d_list_sorted)
        for d in d_list_sorted:
            a = agg[d]
            print(
                f"Expression={Expression} d={d:>3} | "
                f"train_mse={a['train_mean']:.6e}±{a['train_std']:.6e} | "
                f"test_mse={a['test_mean']:.6e}±{a['test_std']:.6e}"
            )
        _plot_errorbars_uniform_x(Expression=Expression, agg=agg, d_list=d_list_sorted)

    total_s = time.perf_counter() - t0_all
    print(f"\nAll done in {total_s:.2f} s")


if __name__ == "__main__":
    main()


Opening HDF5: scalability_experiment_data.h5
d_list = [2, 4, 8, 16, 32, 64]
Expressions = ['P2']
seeds (runs) = [0, 1, 2, 3, 4] (runs_per_d=5)
report_csv_path = reports/operon_scalability_poly2.csv

===== Expression: P2 =====
Loading data for Expression=P2, d=2 ...
Loaded: Xtr=(1000, 2), ytr=(1000,), Xte=(1000, 2), yte=(1000,)

RUN START | Expression=P2 | d=2 | run=1/5 | seed=0
  X_train: (1000, 2) | X_test: (1000, 2)
  operon_cfg: gens=10000, pop=500, max_len=50, max_depth=10, allowed_symbols='add,sub,mul,square,constant,variable', objectives=['r2', 'length'], select='minimum_description_length'
fit() started ...
fit() finished
expr_str: ((-3.493793) + (1.139607 * ((((((1.136412 * X1) ^ 2) + (((0.658748 * X2) + 0.693147) ^ 2)) + ((((-0.897424) * X1) - (((-0.571037) * X1) - ((-0.697091) * X2))) ^ 2)) + ((-0.428957) * X1)) + (1.414214 * X2))))
RESULT | Expression=P2 | d=  2 | run=1/5 | seed=0 | train_mse=1.963546e-02 | test_mse=3.860638e-01 | run_time_s=6.23

RUN START | Expression=P2 |

KeyboardInterrupt: 

In [ ]:
help(SymbolicRegressor)